# Z.AI layout parsing inspection

This notebook sends one image URL from GitHub to `client.layout_parsing.create(...)` using `glm-ocr`, then prints and inspects the structured response so you can verify what the OCR endpoint returns.

In [5]:
from __future__ import annotations

import json
import os
from pathlib import Path

from IPython.display import Image, display
from zai import ZaiClient


In [6]:
def load_env_file(env_path: Path = Path('.env')) -> None:
    if not env_path.exists():
        return

    for raw_line in env_path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue

        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


def github_raw_url(filename: str) -> str:
    return f'https://raw.githubusercontent.com/ikhwanh/zai-api-call/master/images/charges/{filename}'


In [7]:
AVAILABLE_INPUTS = [
    github_raw_url('1077-receipt.jpg'),
    github_raw_url('invoice_cn_taxi.jpg'),
    github_raw_url('invoice_cn_vat.jpg'),
]

MODEL_NAME = 'glm-ocr'

load_env_file()
api_key = os.getenv('ZAI_API_KEY') or os.getenv('API_KEY')
if not api_key:
    raise ValueError('Set ZAI_API_KEY or API_KEY in the environment or .env before running this notebook.')

client = ZaiClient(api_key=api_key)


In [8]:
image_url = AVAILABLE_INPUTS[0]

print(f'Image URL: {image_url}')
display(Image(url=image_url))

response = client.layout_parsing.create(
    model=MODEL_NAME,
    file=image_url,
)

response


Image URL: https://raw.githubusercontent.com/ikhwanh/zai-api-call/master/images/charges/1077-receipt.jpg


LayoutParsingResp(id='2026030414024953ee38a9945c4c9f', created=1772604208, model='glm-ocr', md_results='NAANCHING\n\n103 MONTGOMERRY ST\n\nJERSEY CITY, NJ 07302\n\n2019840709\n\n<div align="center">\n\n# ORDER: SECOND FLOOR 19 Dine-in\n\n</div>\n\nCashier: Kiran\n\n24-Mar-2019 7:14:55P\n\n1 Chicken Lollipop\n\n$9.00\n\nMed $0.00\n\n1 Thai Fried Rice\n\n$12.00\n\nVegetables $0.00\n\nMed $0.00\n\n1 Chicken Tikka Masala $16.00\n\nMILD PLEASE\n\n1 Naan $3.00\n\n1 Vegetable Biryani $12.00\n\nMAKE IT MILD PLEASE\n\n1 Mango Lassi $4.00\n\nSubtotal $56.00\n\nTax $3.71\n\nService Charge (18.0%) $10.08\n\nTotal $69.79', layout_details=[[LayoutDetail(index=0, label='text', bbox_2d=[248.0, 1.0, 524.0, 93.0], content='NAANCHING\n\n103 MONTGOMERRY ST\n\nJERSEY CITY, NJ 07302\n\n2019840709', height=1000, width=867, native_label='text'), LayoutDetail(index=1, label='text', bbox_2d=[131.0, 107.0, 639.0, 192.0], content='<div align="center">\n\n# ORDER: SECOND FLOOR 19 Dine-in\n\n</div>', height=1000, w

In [9]:
payload = response.to_dict(mode='json', exclude_none=True)
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "id": "2026030414024953ee38a9945c4c9f",
  "created": 1772604208,
  "model": "glm-ocr",
  "md_results": "NAANCHING\n\n103 MONTGOMERRY ST\n\nJERSEY CITY, NJ 07302\n\n2019840709\n\n<div align=\"center\">\n\n# ORDER: SECOND FLOOR 19 Dine-in\n\n</div>\n\nCashier: Kiran\n\n24-Mar-2019 7:14:55P\n\n1 Chicken Lollipop\n\n$9.00\n\nMed $0.00\n\n1 Thai Fried Rice\n\n$12.00\n\nVegetables $0.00\n\nMed $0.00\n\n1 Chicken Tikka Masala $16.00\n\nMILD PLEASE\n\n1 Naan $3.00\n\n1 Vegetable Biryani $12.00\n\nMAKE IT MILD PLEASE\n\n1 Mango Lassi $4.00\n\nSubtotal $56.00\n\nTax $3.71\n\nService Charge (18.0%) $10.08\n\nTotal $69.79",
  "layout_details": [
    [
      {
        "index": 0,
        "label": "text",
        "bbox_2d": [
          248.0,
          1.0,
          524.0,
          93.0
        ],
        "content": "NAANCHING\n\n103 MONTGOMERRY ST\n\nJERSEY CITY, NJ 07302\n\n2019840709",
        "height": 1000,
        "width": 867,
        "native_label": "text"
      },
      {
        "ind

In [10]:
usage = payload.get('usage') or {}
data_info = payload.get('data_info') or {}
layout_details = payload.get('layout_details') or []
md_results = payload.get('md_results') or ''

print('Request ID:', payload.get('request_id'))
print('Response ID:', payload.get('id'))
print('Model:', payload.get('model'))
print('Prompt tokens:', usage.get('prompt_tokens'))
print('Completion tokens:', usage.get('completion_tokens'))
print('Total tokens:', usage.get('total_tokens'))
print('Page count:', data_info.get('num_pages'))
print('Layout pages returned:', len(layout_details))
print()
print('Markdown preview:')
print(md_results[:2000])


Request ID: 2026030414024953ee38a9945c4c9f
Response ID: 2026030414024953ee38a9945c4c9f
Model: glm-ocr
Prompt tokens: 897
Completion tokens: 234
Total tokens: 1131
Page count: 1
Layout pages returned: 1

Markdown preview:
NAANCHING

103 MONTGOMERRY ST

JERSEY CITY, NJ 07302

2019840709

<div align="center">

# ORDER: SECOND FLOOR 19 Dine-in

</div>

Cashier: Kiran

24-Mar-2019 7:14:55P

1 Chicken Lollipop

$9.00

Med $0.00

1 Thai Fried Rice

$12.00

Vegetables $0.00

Med $0.00

1 Chicken Tikka Masala $16.00

MILD PLEASE

1 Naan $3.00

1 Vegetable Biryani $12.00

MAKE IT MILD PLEASE

1 Mango Lassi $4.00

Subtotal $56.00

Tax $3.71

Service Charge (18.0%) $10.08

Total $69.79
